# Phase 2b: Re-score without rounding (the threat check)

No training. Loads the fine-tuned model from your Hugging Face repo and the base model, runs them directly (bypassing Laya's 3-decimal rounding), and scores with full precision.

**Settings:** Accelerator **GPU T4 ×2** · Internet **On** · Add Input → **Toxic Comment Classification Challenge** · Add-ons → Secrets → attach **HF_TOKEN** (delete the auto-inserted snippet).

Run All. `EVAL_SET="sample"` takes ~25–35 min for both models.

In [3]:
# ==== CONFIG ====
EVAL_SET = "sample"        # "sample" = same 10k comments as before | "full" = all ~64k test comments (~1 h per model)
SCORE_ZERO_SHOT = True     # also re-score the base model the same way (fair comparison)
FT_REPO = "Vinay57/laya-jigsaw-moderation"
BATCH = 64

In [4]:
!pip install -q -U laya
import os, glob, json, time, numpy as np, pandas as pd, torch, laya
print("laya", laya.__version__, "| CUDA", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 5.1 MB/s eta 0:00:00
laya 0.3.20 | CUDA True


## 1. Same data and test sample as before

In [5]:
LABELS = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']
QUESTIONS = {
  'toxic':         {'type':'noul','instructions':'Is this comment rude, disrespectful or unreasonable?'},
  'severe_toxic':  {'type':'noul','instructions':'Is this comment extremely hateful, aggressive or disrespectful?'},
  'obscene':       {'type':'noul','instructions':'Does this comment contain obscene or vulgar language?'},
  'threat':        {'type':'noul','instructions':'Does this comment threaten to harm someone?'},
  'insult':        {'type':'noul','instructions':'Does this comment insult a person?'},
  'identity_hate': {'type':'noul','instructions':'Does this comment attack someone for their identity (race, religion, gender, sexuality, etc.)?'},
}
MAX_CHARS = 1200

BASE = os.path.dirname(glob.glob('/kaggle/input/**/train.csv*', recursive=True)[0])
def load(name):
    for ext in ['.csv', '.csv.zip']:
        p = f'{BASE}/{name}{ext}'
        if os.path.exists(p): return pd.read_csv(p)
    raise FileNotFoundError(name)
test = load('test').merge(load('test_labels'), on='id')
test = test[test['toxic'] != -1].reset_index(drop=True)
eval_df = test.sample(10000, random_state=42) if EVAL_SET == "sample" else test
print(len(eval_df), "comments | positives:", eval_df[LABELS].sum().to_dict())

10000 comments | positives: {'toxic': 981, 'severe_toxic': 58, 'obscene': 603, 'threat': 34, 'insult': 580, 'identity_hate': 125}


## 2. Direct full-precision scoring

Instead of Laya's rounded probability, we keep the raw model score (the gap between its "yes" and "no" logits).
ROC-AUC only cares about ranking, so this removes the ties created by rounding.

In [6]:
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient
from safetensors.torch import load_file
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config
from laya.common import build_model, build_sequence, QTYPES

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
NOUL = QTYPES["noul"]

def load_raw(repo):
    d = snapshot_download(repo, token=HF_TOKEN,
                          ignore_patterns=["multilingual/*", "typed-decisions/*", "eval/*", "assets/*"])
    _fix_tokenizer_config(d)
    tok = AutoTokenizer.from_pretrained(os.path.join(d, "tokenizer"))
    cfg = json.load(open(os.path.join(d, "rl_agent_config.json")))
    model = build_model(cfg, encoder_dir=os.path.join(d, "encoder"))
    model.load_state_dict(load_file(os.path.join(d, "model.safetensors")), strict=True)
    return model.to("cuda").eval(), tok, cfg, d

def noul_temp(cfg):
    t = cfg.get("temperature", 1.0)
    if isinstance(t, (list, tuple)): return float(t[NOUL])
    if isinstance(t, dict): return float(t.get("noul", 1.0))
    return float(t)

def collate(items, pad_id):
    n, L = len(items), max(len(it[0]) for it in items)
    kmax = max(len(it[1]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long); att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long); mmask = torch.zeros((n, kmax), dtype=torch.bool)
    for i, (seq, mk) in enumerate(items):
        ids[i, :len(seq)] = torch.tensor(seq); att[i, :len(seq)] = 1
        mpos[i, :len(mk)] = torch.tensor(mk); mmask[i, :len(mk)] = True
    return ids, att, mpos, mmask, torch.full((n,), NOUL, dtype=torch.long)

def build_all(tok, cfg, df):
    seqs = []
    for i, text in enumerate(df.comment_text.tolist()):
        state = {'comment': text[:MAX_CHARS]}
        for j, l in enumerate(LABELS):
            ids, mk = build_sequence(tok, state, {"t": "noul", "ins": QUESTIONS[l]["instructions"], "crit": {}},
                                     cfg["max_len"], cfg["head_max_len"])
            seqs.append((i, j, ids, mk))
    return seqs

@torch.no_grad()
def score_all(model, tok, cfg, seqs, n_rows):
    order = sorted(range(len(seqs)), key=lambda k: len(seqs[k][2]))   # similar lengths together = less padding
    margin = np.zeros((n_rows, len(LABELS)))
    torch.cuda.synchronize(); t0 = time.time()
    for b in range(0, len(order), BATCH):
        idx = order[b:b + BATCH]
        ids, att, mpos, mmask, qt = collate([(seqs[k][2], seqs[k][3]) for k in idx], tok.pad_token_id)
        with torch.autocast("cuda", dtype=torch.float16):
            logits, _ = model(ids.cuda(), att.cuda(), mpos.cuda(), mmask.cuda(), qt.cuda())
        lg = logits.float()[:, :2].cpu().numpy()          # option order is [false, true]
        for r, k in enumerate(idx):
            margin[seqs[k][0], seqs[k][1]] = lg[r, 1] - lg[r, 0]
    torch.cuda.synchronize(); secs = time.time() - t0
    prob = 1 / (1 + np.exp(-margin / noul_temp(cfg)))      # calibrated P(yes)
    return margin, prob, secs

## 3. Evaluate

In [7]:
from sklearn.metrics import roc_auc_score
from laya.common import ece_score

def evaluate(name, repo):
    print(f"\n===== {name} ({repo}) =====")
    model, tok, cfg, d = load_raw(repo)
    t0 = time.time(); seqs = build_all(tok, cfg, eval_df); print(f"built {len(seqs)} sequences in {time.time()-t0:.0f}s")
    margin, prob, secs = score_all(model, tok, cfg, seqs, len(eval_df))

    # sanity check: our direct scores must match the official Agent (to its 3-decimal rounding)
    agent = laya.Agent(d, device="cuda")
    chk = agent.predict_batch([{'comment': t[:MAX_CHARS]} for t in eval_df.comment_text.iloc[:100]], QUESTIONS, batch_size=16)
    ag = np.array([[float(r["answers"][l]["noul"]) for l in LABELS] for r in chk])
    diff = np.abs(ag - prob[:100]).max()
    print(f"sanity check vs official Agent: max diff {diff:.4f} " + ("OK" if diff < 0.002 else "MISMATCH, check the scoring"))
    del agent; torch.cuda.empty_cache()

    y = eval_df[LABELS].values
    rows = []
    for j, l in enumerate(LABELS):
        r3 = np.round(prob[:, j], 3)
        rows.append({"label": l,
                     "AUC_rounded(old way)": roc_auc_score(y[:, j], r3),
                     "AUC_full_precision": roc_auc_score(y[:, j], margin[:, j]),
                     "neg_tied_at_0.000": f"{(r3[y[:, j] == 0] == 0).mean():.0%}",
                     "pos_tied_at_0.000": f"{(r3[y[:, j] == 1] == 0).mean():.0%}"})
    tab = pd.DataFrame(rows).set_index("label")
    tab.loc["mean", ["AUC_rounded(old way)", "AUC_full_precision"]] = tab[["AUC_rounded(old way)", "AUC_full_precision"]].mean()
    p, yy = prob.ravel(), y.ravel()
    ece = ece_score(np.maximum(p, 1 - p), ((p >= 0.5) == (yy == 1)).astype(float))
    print(tab.round(4).to_string())
    print(f"ECE {ece:.4f} | temperature {noul_temp(cfg):.3f} | speed {len(eval_df)/secs:.1f} comments/sec (direct batched, 6 q/comment)")

    pd.DataFrame(margin, columns=LABELS, index=eval_df.index).to_csv(f"/kaggle/working/{name}_margins_{EVAL_SET}.csv")
    del model; torch.cuda.empty_cache()
    return {"model": name, "mean_auc_full": float(tab.loc["mean", "AUC_full_precision"]),
            "mean_auc_rounded": float(tab.loc["mean", "AUC_rounded(old way)"]),
            "ece": float(ece), "comments_per_sec": len(eval_df) / secs}

summary = [evaluate("laya_finetuned", FT_REPO)]
if SCORE_ZERO_SHOT:
    summary.append(evaluate("laya_zeroshot", "convaiinnovations/laya"))


===== laya_finetuned (Vinay57/laya-jigsaw-moderation) =====


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

built 60000 sequences in 34s


W0925 20:38:38.125000 58 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


sanity check vs official Agent: max diff 0.0025 MISMATCH, check the scoring
               AUC_rounded(old way)  AUC_full_precision neg_tied_at_0.000 pos_tied_at_0.000
label                                                                                      
toxic                        0.9644              0.9648                0%                0%
severe_toxic                 0.9868              0.9869                0%                0%
obscene                      0.9776              0.9795                0%                0%
threat                       0.9597              0.9496                0%                0%
insult                       0.9809              0.9811                0%                0%
identity_hate                0.9849              0.9868                0%                0%
mean                         0.9757              0.9748               NaN               NaN
ECE 0.0134 | temperature 1.241 | speed 22.7 comments/sec (direct batched, 6 q/comment)

===== la

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

built 60000 sequences in 36s


/tmp/ipykernel_58/3340810386.py:11: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  agent = laya.Agent(d, device="cuda")


sanity check vs official Agent: max diff 0.0018 OK
               AUC_rounded(old way)  AUC_full_precision neg_tied_at_0.000 pos_tied_at_0.000
label                                                                                      
toxic                        0.9589              0.9589                1%                0%
severe_toxic                 0.9764              0.9764               11%                0%
obscene                      0.9555              0.9555               20%                0%
threat                       0.9950              0.9950               27%                0%
insult                       0.9738              0.9738                2%                0%
identity_hate                0.9709              0.9709               17%                0%
mean                         0.9718              0.9718               NaN               NaN
ECE 0.0635 | temperature 1.983 | speed 22.8 comments/sec (direct batched, 6 q/comment)


## 4. Scoreboard

In [8]:
board = pd.DataFrame(summary)
print(board.round(4).to_string(index=False))
if EVAL_SET == "sample":
    print("\nReference on the same 10k sample: Detoxify 0.9854 | TF-IDF 0.9769")
else:
    print("\nReference on full test: Detoxify 0.9863 | TF-IDF 0.9786")
board.to_csv(f"/kaggle/working/rescore_summary_{EVAL_SET}.csv", index=False)

         model  mean_auc_full  mean_auc_rounded    ece  comments_per_sec
laya_finetuned         0.9748            0.9757 0.0134           22.6550
 laya_zeroshot         0.9718            0.9718 0.0635           22.8353

Reference on the same 10k sample: Detoxify 0.9854 | TF-IDF 0.9769


In [9]:
from sklearn.metrics import roc_auc_score
pd.set_option('display.max_colwidth', 200)
ft = pd.read_csv("/kaggle/working/laya_finetuned_margins_sample.csv", index_col=0)
zs = pd.read_csv("/kaggle/working/laya_zeroshot_margins_sample.csv", index_col=0)
y = eval_df.loc[ft.index, LABELS]
zn = lambda m: (m - m.mean()) / m.std()   # put both models on the same scale before blending

t = pd.DataFrame({l: {"fine_tuned": roc_auc_score(y[l], ft[l]),
                      "zero_shot":  roc_auc_score(y[l], zs[l]),
                      "blend_50_50": roc_auc_score(y[l], zn(ft[l]) + zn(zs[l]))} for l in LABELS}).T
t.loc["mean"] = t.mean()
print(t.round(4).to_string())
print("\nMix (threat from zero-shot, rest fine-tuned):",
      round((t.loc[LABELS, "fine_tuned"].sum() - t.loc["threat", "fine_tuned"] + t.loc["threat", "zero_shot"]) / 6, 4))

# The threats fine-tuned ranks worst: pct = share of normal comments it scores this threat above
negf, negz = ft.loc[y.threat == 0, "threat"].values, zs.loc[y.threat == 0, "threat"].values
pos = y.index[y.threat == 1]
rep = pd.DataFrame({"ft_pct": [(negf < ft.at[i, "threat"]).mean() for i in pos],
                    "zs_pct": [(negz < zs.at[i, "threat"]).mean() for i in pos],
                    "text": [eval_df.at[i, "comment_text"][:200].replace("\n", " ") for i in pos]}, index=pos)
print("\nWorst-ranked threats:\n", rep.sort_values("ft_pct").head(8).round(3).to_string())

               fine_tuned  zero_shot  blend_50_50
toxic              0.9648     0.9589       0.9701
severe_toxic       0.9869     0.9764       0.9862
obscene            0.9795     0.9555       0.9780
threat             0.9496     0.9950       0.9970
insult             0.9811     0.9738       0.9814
identity_hate      0.9868     0.9709       0.9891
mean               0.9748     0.9718       0.9836

Mix (threat from zero-shot, rest fine-tuned): 0.9823

Worst-ranked threats:
        ft_pct  zs_pct                                                                                                                                                                                                      text
35601   0.002   0.989                                                                                                                                                                                         shov it up ur ass
43849   0.345   0.991  spsmack my bitch up smack my bitch up smack my bitc